In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from src.evaluation import evaluate_results

RESULTS_DIR = Path("../results")
YEARS = [2025, 2026]

METHODS = [
    "BM25", "TF-IDF",
    "P", "R", "H",
    "Equal-RP", "Equal-HP", "Equal-RH", "Equal-RHP",
    "ISR-RHP", "RRF-RHP"
]

METRICS = ["nDCG@5", "nDCG@10", "MRR"]

all_results = []

for year in YEARS:
    print(f"\nTREC {year}")

    # Load relevance judgments
    qrels = pd.read_csv(
        f"trec_mllm_{year}/data/llm_dev_qrels.txt",
        sep=r"\s+",
        header=None,
        names=["query_id", "iteration", "llm_id", "relevance"],
        dtype={"query_id": str, "llm_id": str}
    )
    qrels = qrels.drop(columns=["iteration"])

    # Read and evaluate saved TREC runs
    for method in METHODS:
        run_path = RESULTS_DIR / str(year) / "runs" / f"{method}.txt"
        run = pd.read_csv(
            run_path,
            sep=r"\s+",
            header=None,
            names=["query_id", "Q0", "llm_id", "rank", "score", "run_tag"],
            dtype={"query_id": str, "llm_id": str}
        )

        # Evaluate saved rankings
        scores = evaluate_results(run[["query_id", "llm_id", "rank", "score"]], qrels)
        all_results.append({
            "Year": year,
            "Method": method,
            "nDCG@5": scores["nDCG@5"],
            "nDCG@10": scores["nDCG@10"],
            "MRR": scores["RR"],
        })

        print(f"{method:12s} | nDCG@5: {scores['nDCG@5']:.4f} | nDCG@10: {scores['nDCG@10']:.4f} | MRR: {scores['RR']:.4f}")

results = pd.DataFrame(all_results)
display(results.round(4))


TREC 2025
BM25         | nDCG@5: 0.3992 | nDCG@10: 0.3931 | MRR: 0.6824
TF-IDF       | nDCG@5: 0.2441 | nDCG@10: 0.2304 | MRR: 0.5136
P            | nDCG@5: 0.3568 | nDCG@10: 0.3382 | MRR: 0.6657
R            | nDCG@5: 0.4821 | nDCG@10: 0.4683 | MRR: 0.7564
H            | nDCG@5: 0.4053 | nDCG@10: 0.3859 | MRR: 0.7001
Equal-RP     | nDCG@5: 0.4761 | nDCG@10: 0.4635 | MRR: 0.7457
Equal-HP     | nDCG@5: 0.4146 | nDCG@10: 0.3962 | MRR: 0.7070
Equal-RH     | nDCG@5: 0.4703 | nDCG@10: 0.4635 | MRR: 0.7670
Equal-RHP    | nDCG@5: 0.4789 | nDCG@10: 0.4656 | MRR: 0.7659
ISR-RHP      | nDCG@5: 0.4286 | nDCG@10: 0.4085 | MRR: 0.7315
RRF-RHP      | nDCG@5: 0.4777 | nDCG@10: 0.4601 | MRR: 0.7632

TREC 2026
BM25         | nDCG@5: 0.2002 | nDCG@10: 0.2324 | MRR: 0.2739
TF-IDF       | nDCG@5: 0.2331 | nDCG@10: 0.2735 | MRR: 0.3089
P            | nDCG@5: 0.3299 | nDCG@10: 0.3594 | MRR: 0.4048
R            | nDCG@5: 0.3197 | nDCG@10: 0.3481 | MRR: 0.3840
H            | nDCG@5: 0.3395 | nDCG@10: 0.3718 

,Year,Method,nDCG@5,nDCG@10,MRR
0,2025,BM25,0.3992,0.3931,0.6824
1,2025,TF-IDF,0.2441,0.2304,0.5136
2,2025,P,0.3568,0.3382,0.6657
3,2025,R,0.4821,0.4683,0.7564
4,2025,H,0.4053,0.3859,0.7001
5,2025,Equal-RP,0.4761,0.4635,0.7457
6,2025,Equal-HP,0.4146,0.3962,0.7070
7,2025,Equal-RH,0.4703,0.4635,0.7670
8,2025,Equal-RHP,0.4789,0.4656,0.7659
9,2025,ISR-RHP,0.4286,0.4085,0.7315


In [2]:
TABLE_ROWS = [
    ("Individual Retrieval Approaches", None, ""),
    ("Baseline (B1)", "BM25", "BoW (Okapi BM25)"),
    ("Baseline (B2)", "TF-IDF", "BoW (TF-IDF)"),
    ("Median of (B1, B2)", "baseline", "BoW"),
    ("Improved Profile (P)", "P", "Text embedding"),
    ("Response Retrieval (R)", "R", "Text embedding"),
    ("Hierarchical Retrieval (H)", "H", "Text embedding"),
    ("Median of (P, R, H)", "individual", "Text embedding"),

    ("Aggregation Retrieval Approaches", None, ""),
    ("Equal (P + R)", "Equal-RP", "Text embedding"),
    ("Equal (P + H)", "Equal-HP", "Text embedding"),
    ("Equal (R + H)", "Equal-RH", "Text embedding"),
    ("Equal (P + R + H)", "Equal-RHP", "Text embedding"),
    ("ISR (P + R + H)", "ISR-RHP", "Text embedding"),
    ("RRF (P + R + H)", "RRF-RHP", "Text embedding"),
    ("Median of (ISR, RRF, Equal)", "aggregation", "Text embedding"),
]

tables = {}

for year in YEARS:
    scores = (results[results["Year"] == year].set_index("Method")[METRICS].reindex(METHODS))
    scores = scores.round(3)
    baseline = scores.loc[["BM25", "TF-IDF"]].median().round(3)

    medians = {
        "baseline": baseline,
        "individual": scores.loc[["P", "R", "H"]].median(),
        "aggregation": scores.loc[
            ["ISR-RHP", "RRF-RHP", "Equal-RHP"]
        ].median()
    }

    rows = []
    for label, method, text_model in TABLE_ROWS:
        row = {"Approach": label, "Text Model": text_model}

        if method is not None:
            values = medians[method] if method in medians else scores.loc[method]

            for metric in METRICS:
                value = round(float(values[metric]), 3)
                median = round(float(baseline[metric]), 3)
            
                row[metric] = value
                if method != "baseline": row[f"Δ {metric}"] = (value / median - 1) * 100
        rows.append(row)

    table = pd.DataFrame(rows)[[
        "Approach", "Text Model",
        "nDCG@5", "Δ nDCG@5",
        "nDCG@10", "Δ nDCG@10",
        "MRR", "Δ MRR"
    ]]

    tables[year] = table

    # Highlight best and second-best methods
    def highlight(df):
        styles = pd.DataFrame("", index=df.index, columns=df.columns)
        method_rows = [i for i, (_, method, _) in enumerate(TABLE_ROWS) if method in METHODS]

        for metric in METRICS:
            top = (
                df.loc[method_rows, metric]
                .dropna()
                .drop_duplicates()
                .nlargest(2)
                .tolist()
            )

            for i in method_rows:
                if top and df.loc[i, metric] == top[0]:
                    styles.loc[i, metric] = "font-weight: bold"
                elif len(top) > 1 and df.loc[i, metric] == top[1]:
                    styles.loc[i, metric] = "text-decoration: underline"

        for i, (_, method, _) in enumerate(TABLE_ROWS):
            if method is None:
                styles.loc[i, "Approach"] = "font-weight: bold"
            elif method in medians:
                styles.loc[i, "Approach"] = "font-style: italic"

        return styles

    styled = (
        table.style
        .format({
            "nDCG@5": "{:.3f}",
            "nDCG@10": "{:.3f}",
            "MRR": "{:.3f}",
            "Δ nDCG@5": "{:+.1f}%",
            "Δ nDCG@10": "{:+.1f}%",
            "Δ MRR": "{:+.1f}%"
        }, na_rep="")
        .apply(highlight, axis=None)
        .relabel_index([
            "Approach", "Text Model",
            "nDCG@5", "Δ (%)",
            "nDCG@10", "Δ (%)",
            "MRR", "Δ (%)"
        ], axis="columns")
        .hide(axis="index")
    )

    print(f"\nTREC {year}")
    display(styled)


TREC 2025


Approach,Text Model,nDCG@5,Δ (%),nDCG@10,Δ (%),MRR,Δ (%)
Individual Retrieval Approaches,,,,,,,
Baseline (B1),BoW (Okapi BM25),0.399,+23.9%,0.393,+26.0%,0.682,+14.0%
Baseline (B2),BoW (TF-IDF),0.244,-24.2%,0.230,-26.3%,0.514,-14.0%
"Median of (B1, B2)",BoW,0.322,,0.312,,0.598,
Improved Profile (P),Text embedding,0.357,+10.9%,0.338,+8.3%,0.666,+11.4%
Response Retrieval (R),Text embedding,0.482,+49.7%,0.468,+50.0%,0.756,+26.4%
Hierarchical Retrieval (H),Text embedding,0.405,+25.8%,0.386,+23.7%,0.700,+17.1%
"Median of (P, R, H)",Text embedding,0.405,+25.8%,0.386,+23.7%,0.700,+17.1%
Aggregation Retrieval Approaches,,,,,,,
Equal (P + R),Text embedding,0.476,+47.8%,0.463,+48.4%,0.746,+24.7%



TREC 2026


Approach,Text Model,nDCG@5,Δ (%),nDCG@10,Δ (%),MRR,Δ (%)
Individual Retrieval Approaches,,,,,,,
Baseline (B1),BoW (Okapi BM25),0.200,-7.8%,0.232,-8.3%,0.274,-6.2%
Baseline (B2),BoW (TF-IDF),0.233,+7.4%,0.274,+8.3%,0.309,+5.8%
"Median of (B1, B2)",BoW,0.217,,0.253,,0.292,
Improved Profile (P),Text embedding,0.330,+52.1%,0.359,+41.9%,0.405,+38.7%
Response Retrieval (R),Text embedding,0.320,+47.5%,0.348,+37.5%,0.384,+31.5%
Hierarchical Retrieval (H),Text embedding,0.339,+56.2%,0.372,+47.0%,0.400,+37.0%
"Median of (P, R, H)",Text embedding,0.330,+52.1%,0.359,+41.9%,0.400,+37.0%
Aggregation Retrieval Approaches,,,,,,,
Equal (P + R),Text embedding,0.374,+72.4%,0.413,+63.2%,0.437,+49.7%
